# Glucose dashboard

How much of the time was each patient's glucose in the target range (70–180 mg/dL)?
Choose a patient to see their readings.

In [ ]:
import warnings

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from jupyterhealth_client import Code, JupyterHealthClient

warnings.filterwarnings("ignore")

client = JupyterHealthClient()
study_id = 30011  # Sample Data study

In [ ]:
def show_patient(patient_id):
    df = client.list_observations_df(
        patient_id=patient_id,
        study_id=study_id,
        code=Code.BLOOD_GLUCOSE,
        limit=10_000,
    )
    if df.empty:
        print("No glucose readings for this patient.")
        return
    glucose = pd.DataFrame(
        {
            "time": pd.to_datetime(df["effective_time_frame_date_time_local"]),
            "glucose": df["blood_glucose_value"],
        }
    ).sort_values("time")

    mean_glucose = glucose["glucose"].mean()
    in_range = glucose["glucose"].between(70, 180).mean() * 100
    print(f"Average glucose: {mean_glucose:.0f} mg/dL")
    print(f"Time in range: {in_range:.0f}%")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.axhspan(70, 180, color="green", alpha=0.15, label="Target range")
    ax.plot(glucose["time"], glucose["glucose"], linewidth=0.8)
    ax.set_ylabel("Glucose (mg/dL)")
    ax.legend(loc="upper right")
    plt.show()


patients = [
    (f"{p['nameGiven']} {p['nameFamily']}", p["id"])
    for p in client.list_patients(study_id=study_id)
]
widgets.interact(show_patient, patient_id=widgets.Dropdown(options=patients, description="Patient"));